<a href="https://colab.research.google.com/github/revathigott123/commerceai/blob/main/CommerceAgent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sys, platform
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
try:
    import subprocess
    gpu = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True)
    print("GPU:", gpu.stdout.strip() or "none (CPU only — fine for Stage 1)")
except Exception:
    print("GPU: none detected (CPU only — fine for Stage 1)")


Python: 3.12.13
Platform: Linux-6.6.122+-x86_64-with-glibc2.35
GPU: none detected (CPU only — fine for Stage 1)
Install finished.
Git LFS initialized.
fatal: destination path 'esci-data' already exists and is not an empty directory.


In [ ]:
!pip install -q rank_bm25 pyarrow
print("Install finished.")

In [ ]:
import os

if not os.path.exists("esci-data"):
    !git lfs install --skip-repo
    !git clone --quiet https://github.com/amazon-science/esci-data.git
    !cd esci-data && git lfs pull
    print("Clone complete.")
else:
    print("esci-data already present, skipping clone.")
    !cd esci-data && git lfs pull

esci-data already present, skipping clone.
^C


In [ ]:
base = "esci-data/shopping_queries_dataset/"
needed = [
    "shopping_queries_dataset_examples.parquet",
    "shopping_queries_dataset_products.parquet",
]

ok = True
for f in needed:
    path = base + f
    if not os.path.exists(path):
        print(f"MISSING: {path}")
        ok = False
        continue
    size_mb = os.path.getsize(path) / 1e6
    status = "OK" if size_mb > 1 else "TOO SMALL (LFS placeholder!)"
    if size_mb <= 1:
        ok = False
    print(f"{status:<28} {size_mb:8.1f} MB  {f}")

if not ok:
    print("\n>>> FIX: run this, then re-run this cell:")
    print(">>>   !cd esci-data && git lfs pull")
else:
    print("\nAll data files look good.")

OK                               51.3 MB  shopping_queries_dataset_examples.parquet
OK                             1108.9 MB  shopping_queries_dataset_products.parquet

All data files look good.


In [ ]:
import pandas as pd

examples = pd.read_parquet(base + "shopping_queries_dataset_examples.parquet")
products = pd.read_parquet(base + "shopping_queries_dataset_products.parquet")

print("examples shape:", examples.shape)
print("examples columns:", list(examples.columns))
print()
print("products shape:", products.shape)
print("products columns:", list(products.columns))

expected_examples = ["query", "query_id", "product_id", "esci_label",
                     "product_locale", "small_version", "split"]
expected_products = ["product_id", "product_title", "product_locale"]

missing_e = [c for c in expected_examples if c not in examples.columns]
missing_p = [c for c in expected_products if c not in products.columns]

if missing_e or missing_p:
    print("Missing in examples:", missing_e)
    print("Missing in products:", missing_p)
    print("Compare against the printed column lists above and adjust names.")
else:
    print("All expected columns present. Safe to continue.")

examples shape: (2621288, 9)
examples columns: ['example_id', 'query', 'query_id', 'product_id', 'product_locale', 'esci_label', 'small_version', 'large_version', 'split']

products shape: (1814924, 7)
products columns: ['product_id', 'product_title', 'product_description', 'product_bullet_point', 'product_brand', 'product_color', 'product_locale']
All expected columns present. Safe to continue.


In [ ]:
examples["small_version"] = examples["small_version"].astype(int)

ex = examples[
    (examples["product_locale"] == "us")
    & (examples["small_version"] == 1)
    & (examples["split"] == "test")
].copy()

prod_us = products[products["product_locale"] == "us"]
df = ex.merge(prod_us[["product_id", "product_title"]], on="product_id", how="left")
df = df.dropna(subset=["product_title"])

print("rows after filtering:", len(df))
print("unique queries:", df["query_id"].nunique())
df[["query", "product_title", "esci_label"]].head(5)

rows after filtering: 181701
unique queries: 8956


,query,product_title,esci_label
0,!qscreen fence without holes,FOTMISHU 6Pcs Greenhouse Hoops Rust-Free Grow ...,I
1,!qscreen fence without holes,Zippity Outdoor Products ZP19028 Unassembled M...,I
2,!qscreen fence without holes,Zippity Outdoor Products ZP19026 Lightweight P...,E
3,!qscreen fence without holes,ColourTree 4' x 50' Green Fence Privacy Screen...,S
4,!qscreen fence without holes,ColourTree 6' x 50' Black Fence Privacy Screen...,S


In [ ]:
label_to_gain = {"E": 3, "S": 2, "C": 1, "I": 0}
df["gain"] = df["esci_label"].map(label_to_gain)
df["relevant"] = df["esci_label"].isin(["E", "S"]).astype(int)
df["esci_label"].value_counts()

,count
esci_label,
E,79708
S,63563
I,30331
C,8099


In [ ]:
SAMPLE_QUERIES = 300 #can be changed to test out bigger numbers

query_ids = df["query_id"].drop_duplicates().sample(
    n=min(SAMPLE_QUERIES, df["query_id"].nunique()), random_state=42)
df = df[df["query_id"].isin(query_ids)].reset_index(drop=True)
print("queries in this run:", df["query_id"].nunique())
print("candidate rows:", len(df))

queries in this run: 300
candidate rows: 6030


In [ ]:
from rank_bm25 import BM25Okapi

def tokenize(text):
    return str(text).lower().split()

def bm25_rank_one_query(group):
    titles = group["product_title"].tolist()
    tokenized = [tokenize(t) for t in titles]
    bm25 = BM25Okapi(tokenized)
    scores = bm25.get_scores(tokenize(group["query"].iloc[0]))
    ranked = group.copy()
    ranked["bm25_score"] = scores
    return ranked.sort_values("bm25_score", ascending=False)

In [ ]:
import numpy as np
from sklearn.metrics import ndcg_score

def ndcg_at_k(ranked, k=10):
    gains = ranked["gain"].to_numpy()

    if gains.sum() == 0:
        return None

    true = gains.reshape(1, -1)
    pred = np.arange(len(gains), 0, -1).reshape(1, -1)

    return ndcg_score(true, pred, k=k)


def mrr(ranked):
    rels = ranked["relevant"].to_numpy()
    hit = np.where(rels == 1)[0]
    return 1.0 / (hit[0] + 1) if len(hit) else 0.0


def precision_at_k(ranked, k=5):
    rels = ranked["relevant"].to_numpy()[:k]
    return rels.sum() / len(rels) if len(rels) > 0 else 0.0

In [ ]:
results = []

for qid, group in df.groupby("query_id", group_keys=False):
    ranked = bm25_rank_one_query(group)
    results.append({
        "query_id": qid,
        "ndcg@10": ndcg_at_k(ranked, k=10),
        "mrr": mrr(ranked),
        "precision@5": precision_at_k(ranked, k=5),
    })

results_df = pd.DataFrame(results)

print("Queries evaluated:", len(results_df))
print("Queries with usable NDCG:", results_df["ndcg@10"].notna().sum())
print()
print("Mean NDCG@10:    ", results_df["ndcg@10"].dropna().mean())
print("Mean MRR:        ", results_df["mrr"].mean())
print("Mean Precision@5:", results_df["precision@5"].mean())

results_df.head(10)

Queries evaluated: 300
Queries with usable NDCG: 300

Mean NDCG@10:     0.810735480049913
Mean MRR:         0.9004577922077921
Mean Precision@5: 0.8373333333333332


,query_id,ndcg@10,mrr,precision@5
0,491,0.576590,1.0,0.6
1,553,0.646856,0.5,0.8
2,599,0.404650,0.5,0.4
3,600,0.898255,1.0,1.0
4,731,0.491797,0.5,0.6
5,954,0.755535,1.0,0.6
6,1213,0.963318,1.0,1.0
7,1246,0.971619,1.0,1.0
8,1335,0.964216,1.0,1.0
9,1503,0.798593,1.0,1.0


In [ ]:
# Look up any query_id from the results table
qid_to_check = 600

group = df[df["query_id"] == qid_to_check]
ranked = bm25_rank_one_query(group)

print("Query:", ranked["query"].iloc[0])
print()
print(ranked[["product_title", "esci_label", "bm25_score"]].head(10))

Query: 0sxk ofrece fidget toys not expensive

                                         product_title esci_label  bm25_score
120  Fidget Toys and Sensory Toys by BUNMO - Textur...          S    2.465073
133  Figetget Toys Fidget Toys Pack - Fidgets Box I...          E    2.444911
122  Fidget Toys and Textured Sensory Toys by BUNMO...          E    2.326745
153  EDsportshouse Sensory Toys Bundle-Stress Relie...          E    2.170102
130  61 Pcs Sensory Fidget Toys Pack,Stress & Anxie...          S    2.168365
155  ASSBABY Fidget Toys - Funny Facial Expressions...          E    2.114762
124  Sensory Fidget Toys 23-Pack – Stress Relief To...          E    2.038718
157  Sensory Toys Set 38 Pack, Stress Relief Fidget...          E    2.012138
152  GONGYIHONG 40 Pack Sensory Fidget Toys Bundle,...          E    1.975388
131  HANGYUAN Push Popping Bubble Sensory Fidget To...          E    1.887651
